In [1]:
import numpy as np
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "mrpc"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 3668
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation": raw_datasets['validation'],
    "test": raw_datasets['test']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

In [10]:
raw_train_dataset[5]['sentence1'], raw_train_dataset[5]['sentence2']

('Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .',
 "With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .")

In [11]:
raw_train_dataset[5]['label']

1

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=128, collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=128, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=128, collate_fn=data_collator
)

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([128]),
 'input_ids': torch.Size([128, 100]),
 'token_type_ids': torch.Size([128, 100]),
 'attention_mask': torch.Size([128, 100])}

# 4. Load Model

In [19]:
import sys
sys.path.insert(0, '../../')

In [20]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [21]:
# id2label, label2id dicts for the outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [22]:
model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
model.set_use_module_grafting(False)
model.set_use_scc_status(False)

In [24]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(1.1277, grad_fn=<NllLossBackward0>) torch.Size([128, 2])


In [25]:
model_6 = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [26]:
model_6.set_use_module_grafting(False)
model_6.set_use_scc_status(False)

In [27]:
outputs = model_6(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(0.6733, grad_fn=<NllLossBackward0>) torch.Size([128, 2])


In [28]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
model_6.to(device)

device

device(type='cuda')

### Load Trained Weights

In [29]:
load_path = '../../glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt', map_location=device)
model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [30]:
load_graft_path = '../../similar_layer_merge/weights/'
graft_state = torch.load(load_graft_path + f'layer-6-{task_name}.pt', map_location=device)

In [31]:
from merge_helper import apply_layer_tracking_to_model

In [32]:
model_6 = apply_layer_tracking_to_model(model_6, graft_state['layer_track'])

In [33]:
model_6.load_state_dict(graft_state['model_state_dict'])

<All keys matched successfully>

In [34]:
from train_eval_func import eval_loop

In [35]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

In [36]:
eval_loop(model_6, val_dataloader, task_name, device)[0]

{'accuracy': 0.8480392156862745, 'f1': 0.8938356164383562}

# 5 Speed Test

In [37]:
import numpy as np
from time import perf_counter

def time_pipeline(model, batch):
    latencies = []
    
    # Warmup
    for _ in range(10):
        _ = model(**batch)
        
    # Timed run
    for _ in range(100):
        start_time = perf_counter()
        model(**batch)
        latency = perf_counter() - start_time
        latencies.append(latency)
        
    # Compute run statistics
    time_avg_ms = 1000 * np.mean(latencies)
    time_std_ms = 1000 * np.std(latencies)
    print(f"Average latency (ms) - {time_avg_ms:.2f} +\- {time_std_ms:.2f}")
    return {"time_avg_ms": time_avg_ms, "time_std_ms": time_std_ms}

In [38]:
from pathlib import Path

def compute_size(model):
    state_dict = model.state_dict()
    tmp_path = Path("model.pt")
    torch.save(state_dict, tmp_path)
    # Calculate size in megabytes
    size_mb = Path(tmp_path).stat().st_size / (1024 * 1024)
    # Delete temporary file
    tmp_path.unlink()
    print(f"Model size (MB) - {size_mb:.2f}")
    return size_mb

In [39]:
def inference_time_on_various_batch_sizes(model, batch, batch_sizes):
    # batch_sizes = [1, 8, 16, 32, 64, 128]
    latencies = []
    
    for batch_size in batch_sizes:
        print("Batch Size:", batch_size)
        modified_batch = {k: v[:batch_size] for k, v in batch.items()}
        latency = time_pipeline(model, batch)
        latencies.append(latency)
    
    return latencies

In [40]:
batch_sizes = [1, 8, 16, 32, 64, 128]

In [44]:
for batch in train_dataloader:
    break
batch = {k: v.to(device) for k, v in batch.items()}
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([128]),
 'input_ids': torch.Size([128, 96]),
 'token_type_ids': torch.Size([128, 96]),
 'attention_mask': torch.Size([128, 96])}

In [41]:
compute_size(model)

Model size (MB) - 417.71


417.7130880355835

In [43]:
compute_size(model_6)

Model size (MB) - 417.58


417.5763692855835

In [45]:
original_inference = inference_time_on_various_batch_sizes(model, batch, batch_sizes)

Batch Size: 1
Average latency (ms) - 311.31 +\- 10.17
Batch Size: 8
Average latency (ms) - 315.60 +\- 11.37
Batch Size: 16
Average latency (ms) - 315.57 +\- 11.02
Batch Size: 32
Average latency (ms) - 314.75 +\- 10.24
Batch Size: 64
Average latency (ms) - 315.09 +\- 10.01
Batch Size: 128
Average latency (ms) - 314.43 +\- 10.46


In [46]:
original_inference = inference_time_on_various_batch_sizes(model_6, batch, batch_sizes)

Batch Size: 1
Average latency (ms) - 300.69 +\- 10.23
Batch Size: 8
Average latency (ms) - 300.48 +\- 10.04
Batch Size: 16
Average latency (ms) - 300.28 +\- 9.71
Batch Size: 32
Average latency (ms) - 301.38 +\- 10.39
Batch Size: 64
Average latency (ms) - 299.99 +\- 9.18
Batch Size: 128
Average latency (ms) - 300.34 +\- 10.05
